**Status: Battle & Event Predictions.** Estimates how likely a chaser is to pass the car ahead on track within the next 5 laps, given the fight stays on track (no pit stop or safety car in the window).

Reads `f1_cleaned_lap_dataset` (pairs) and `f1_master_lap_dataset` (full 20-car grid state). Writes `f1_battle_pairs`, `f1_battle_bands`, `f1_circuit_overtake_prior` and `f1_replay_lap_state`, which the App reads directly. The model ships only if it passes the pre-registered rule in Step 3; otherwise the gap-bucket lookup is what the App shows.

In [ ]:
import os

from pyspark.sql import functions as F

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

from sklearn.metrics import roc_auc_score, brier_score_loss

In [0]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("F1-Battle-Model")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_cleaned_lap_dataset"
df = spark.table(MASTER_PATH)

### Step 1: Battle pairs (causal features, clean-window labels)

In [ ]:
N_LOOKAHEAD = 5
GAP_THRESHOLD = 3.0
FOLD_YEARS = [2021, 2022, 2023]

d = (
    df.select("Year", "Race", "Circuit", "Driver", "LapNumber", "Position", "GapToAhead", "HasPit", "TrackStatus", "TyreLife", "SpeedST")
    .toPandas()
    .drop_duplicates(["Year", "Race", "Driver", "LapNumber"])
)
d["SC"] = d["TrackStatus"].astype(str).str.contains("[45]")

def race_pairs(g):
    pv = lambda c: g.pivot(index="LapNumber", columns="Driver", values=c)
    P, G, T, S = pv("Position"), pv("GapToAhead"), pv("TyreLife"), pv("SpeedST")
    H, Z = pv("HasPit").fillna(0), pv("SC").fillna(False).astype(bool)
    laps = P.index.to_numpy()
    out = []
    for i, lap in enumerate(laps):
        row = P.loc[lap].dropna().sort_values()
        dr = row.index.tolist()
        for a in range(1, len(dr)):
            t, l = dr[a], dr[a - 1]
            gp = G.at[lap, t]
            if int(row[t]) != int(row[l]) + 1 or not (pd.notna(gp) and 0 < gp <= GAP_THRESHOLD):
                continue
            win = laps[i:i + N_LOOKAHEAD + 2]
            clean = not ((H.loc[win, [t, l]].to_numpy() > 0).any() or Z.loc[win, [t, l]].to_numpy().any())
            y = 0
            for k in range(1, N_LOOKAHEAD + 1):
                if i + k >= len(laps):
                    break
                pt, pl = P.at[laps[i + k], t], P.at[laps[i + k], l]
                if pd.notna(pt) and pd.notna(pl) and pt < pl:
                    y = 1
            gv = [x for x in (G.at[laps[j], t] for j in range(max(0, i - 2), i + 1)) if pd.notna(x)]
            closing = (gv[0] - gv[-1]) / (len(gv) - 1) if len(gv) >= 2 else 0.0
            td, sd = T.at[lap, t] - T.at[lap, l], S.at[lap, t] - S.at[lap, l]
            out.append(dict(LapNumber=int(lap), Trailing=t, Leading=l, Gap=float(gp), Closing=float(closing),
                            TyreDelta=0.0 if pd.isna(td) else float(td), SpeedDelta=0.0 if pd.isna(sd) else float(sd),
                            Clean=bool(clean), Y=int(y)))
    return out

rows = []
for (yr, rc), g in d.groupby(["Year", "Race"]):
    for r in race_pairs(g):
        rows.append(dict(Year=int(yr), Race=rc, Circuit=g["Circuit"].iloc[0], **r))
pairs_pd = pd.DataFrame(rows)

clean_pairs = pairs_pd[pairs_pd["Clean"]]
print("Pairs:", len(pairs_pd), "| clean:", len(clean_pairs), "| clean on-track pass rate:", round(clean_pairs["Y"].mean(), 4))
print("Old label (any swap incl. pit/SC) rate:", round(pairs_pd["Y"].mean(), 4))
print(clean_pairs.groupby(pd.cut(clean_pairs["Gap"], [0, .5, 1, 2, 3.01]), observed=False)["Y"].mean().round(4))

### Step 2: Walk-forward models vs baselines

In [ ]:
FEATS = ["Gap", "Closing", "TyreDelta", "SpeedDelta", "CircPrior"]
GAP_BINS = [0, .5, 1, 2, 3.01]

def fit_predict(train, test):
    tr = train[train["Clean"]].copy()
    te = test.copy()
    prior = tr.groupby("Circuit")["Y"].mean()
    base = tr["Y"].mean()
    for x in (tr, te):
        x["CircPrior"] = x["Circuit"].map(prior).fillna(base)
    tab = tr.groupby(pd.cut(tr["Gap"], GAP_BINS), observed=False)["Y"].mean()
    xgb_m = XGBClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, min_child_weight=20, random_state=42).fit(tr[FEATS], tr["Y"])
    return {
        "lookup": pd.cut(te["Gap"], GAP_BINS).map(tab).astype(float).fillna(base).to_numpy(),
        "logit_gap": LogisticRegression().fit(tr[["Gap"]], tr["Y"]).predict_proba(te[["Gap"]])[:, 1],
        "xgb": xgb_m.predict_proba(te[FEATS])[:, 1],
    }

pairs_pd = pairs_pd.reset_index(drop=True)
for k in ("lookup", "logit_gap", "xgb"):
    pairs_pd[k] = np.nan
for fy in range(2018, 2024):
    train = pairs_pd[pairs_pd["Year"] < fy] if fy >= 2021 else pairs_pd[pairs_pd["Year"] >= 2021]
    test_idx = pairs_pd.index[pairs_pd["Year"] == fy]
    preds = fit_predict(train, pairs_pd.loc[test_idx])
    for k, v in preds.items():
        pairs_pd.loc[test_idx, k] = v

ev = pairs_pd[pairs_pd["Clean"] & pairs_pd["Year"].isin(FOLD_YEARS)]
for fy in FOLD_YEARS:
    e = ev[ev["Year"] == fy]
    print(fy, f"n={len(e)} pos={int(e['Y'].sum())}", {k: (round(roc_auc_score(e["Y"], e[k]), 3), round(brier_score_loss(e["Y"], e[k]), 4)) for k in ("lookup", "logit_gap", "xgb")})
print("Pooled", {k: (round(roc_auc_score(ev["Y"], ev[k]), 4), round(brier_score_loss(ev["Y"], ev[k]), 5)) for k in ("lookup", "logit_gap", "xgb")})

### Step 3: Pre-registered ship rule (race-clustered bootstrap)

In [ ]:
rng = np.random.default_rng(0)
y_ev = ev["Y"].to_numpy()
race_ids = (ev["Year"].astype(str) + ev["Race"]).to_numpy()
races = np.unique(race_ids)
by_race = {r: np.where(race_ids == r)[0] for r in races}

def boot(base, model, n=1000):
    d_b, d_a = [], []
    for _ in range(n):
        pk = np.concatenate([by_race[r] for r in rng.choice(races, len(races))])
        yb = y_ev[pk]
        if yb.min() == yb.max():
            continue
        d_b.append(brier_score_loss(yb, ev[base].to_numpy()[pk]) - brier_score_loss(yb, ev[model].to_numpy()[pk]))
        d_a.append(roc_auc_score(yb, ev[model].to_numpy()[pk]) - roc_auc_score(yb, ev[base].to_numpy()[pk]))
    return (np.mean(d_b), *np.percentile(d_b, [2.5, 97.5])), (np.mean(d_a), *np.percentile(d_a, [2.5, 97.5]))

(b_m, b_lo, b_hi), (a_m, a_lo, a_hi) = boot("lookup", "xgb")
print(f"xgb vs lookup: dBrier {b_m:+.5f} [{b_lo:+.5f}, {b_hi:+.5f}] | dAUC {a_m:+.4f} [{a_lo:+.4f}, {a_hi:+.4f}]")
SHIP_MODEL = (
    (b_m >= 0.001 or a_m >= 0.03)
    and b_lo > 0
    and roc_auc_score(y_ev, ev["xgb"]) > roc_auc_score(y_ev, ev["logit_gap"])
)
print("Ship rule passed:", SHIP_MODEL)
pairs_pd["P"] = pairs_pd["xgb"] if SHIP_MODEL else pairs_pd["lookup"]

### Step 4: Bands, circuit prior and pair table

In [ ]:
ev = pairs_pd[pairs_pd["Clean"] & pairs_pd["Year"].isin(FOLD_YEARS)]
cuts = ev["P"].quantile([1 / 3, 2 / 3]).to_numpy()
pairs_pd["Band"] = np.select([pairs_pd["P"] < cuts[0], pairs_pd["P"] < cuts[1]], ["Low", "Medium"], "High")
ev = pairs_pd[pairs_pd["Clean"] & pairs_pd["Year"].isin(FOLD_YEARS)]
bands = (
    ev.groupby("Band")
    .agg(P_Lo=("P", "min"), P_Hi=("P", "max"), ObservedRate=("Y", "mean"), Pairs=("Y", "size"))
    .reindex(["Low", "Medium", "High"])
    .reset_index()
)
print(bands)

circuit_prior_full = (
    pairs_pd[pairs_pd["Clean"]].groupby("Circuit")
    .agg(CircuitOvertakePrior=("Y", "mean"), BattleCount=("Y", "size"))
    .reset_index()
)

keep_cols = ["Year", "Race", "Circuit", "LapNumber", "Trailing", "Leading", "Gap", "Closing", "TyreDelta", "Clean", "Y", "P", "Band"]
spark.createDataFrame(pairs_pd[keep_cols]).write.mode("overwrite").saveAsTable("workspace.default.f1_battle_pairs")
spark.createDataFrame(bands).write.mode("overwrite").saveAsTable("workspace.default.f1_battle_bands")
spark.createDataFrame(circuit_prior_full).write.mode("overwrite").saveAsTable("workspace.default.f1_circuit_overtake_prior")

### Step 5: Full-grid replay state (all cars, including early retirements)

In [ ]:
m = (
    spark.table("workspace.default.f1_master_lap_dataset")
    .select("Year", "Race", "Circuit", "Driver", "DriverNumber", "BroadcastName", "TeamName", "TeamColor", "LapNumber",
            "Position", "Time", "Compound", "TyreLife", "HasPit", "TrackStatus", "QualiPosition")
    .toPandas()
    .drop_duplicates(["Year", "Race", "Driver", "LapNumber"])
    .sort_values(["Year", "Race", "Driver", "LapNumber"])
)
m["SC"] = m["TrackStatus"].astype(str).str.contains("[45]")
m["HasPit"] = m["HasPit"].fillna(0).astype(int)
# gaps from the lap-completion clock (the stored GapToAhead predates the current phase_1 logic)
m = m.sort_values(["Year", "Race", "LapNumber", "Position"])
_g = m.groupby(["Year", "Race", "LapNumber"])
m["GapToAhead"] = np.where(_g["Position"].shift(1) == m["Position"] - 1, m["Time"] - _g["Time"].shift(1), np.nan)
m["DeltaToLeader"] = m["Time"] - _g["Time"].transform("min")
m = m.sort_values(["Year", "Race", "Driver", "LapNumber"])
m["Position"] = m.groupby(["Year", "Race", "Driver"])["Position"].ffill()

grid0 = (
    m.groupby(["Year", "Race", "Driver"], as_index=False).first()
    .sort_values(["Year", "Race", "QualiPosition", "Position"], na_position="last")
    .assign(LapNumber=0, GapToAhead=np.nan, DeltaToLeader=np.nan, HasPit=0, SC=False)
)
grid0["Position"] = grid0.groupby(["Year", "Race"]).cumcount() + 1
lap_state = pd.concat([grid0, m], ignore_index=True)
lap_state["TeamColor"] = lap_state["TeamColor"].fillna("").astype(str)
lap_state = lap_state[["Year", "Race", "Circuit", "Driver", "DriverNumber", "BroadcastName", "TeamName", "TeamColor",
                       "LapNumber", "Position", "GapToAhead", "DeltaToLeader", "Compound", "TyreLife", "HasPit", "SC"]]
lap_state["DriverNumber"] = lap_state["DriverNumber"].astype(str)
spark.createDataFrame(lap_state).write.mode("overwrite").saveAsTable("workspace.default.f1_replay_lap_state")
print("Replay state rows:", len(lap_state), "| races:", lap_state.groupby(["Year", "Race"]).ngroups, "| max cars in a race:", lap_state.groupby(["Year", "Race"])["Driver"].nunique().max())

### Step 6: Track and register the walk-forward result

In [ ]:
mlflow.set_registry_uri("databricks-uc")

final_tr = pairs_pd[pairs_pd["Clean"]].copy()
final_tr["CircPrior"] = final_tr["Circuit"].map(circuit_prior_full.set_index("Circuit")["CircuitOvertakePrior"])
battle_model = XGBClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, min_child_weight=20, random_state=42).fit(final_tr[FEATS], final_tr["Y"])

with mlflow.start_run(run_name="battle_pass_model"):
    mlflow.log_metrics({
        "wf_auc_xgb": roc_auc_score(y_ev, ev["xgb"]), "wf_auc_lookup": roc_auc_score(y_ev, ev["lookup"]),
        "wf_brier_xgb": brier_score_loss(y_ev, ev["xgb"]), "wf_brier_lookup": brier_score_loss(y_ev, ev["lookup"]),
        "wf_dbrier_ci_lo": b_lo, "wf_dauc_ci_lo": a_lo, "ship_model": float(SHIP_MODEL),
    })
    mlflow.xgboost.log_model(
        battle_model, "battle_model",
        signature=infer_signature(final_tr[FEATS], battle_model.predict_proba(final_tr[FEATS])),
        registered_model_name="workspace.default.f1_battle_model",
    )

BATTLE_VOLUME_DIR = "/Volumes/workspace/default/f1_data/battle_models"
os.makedirs(BATTLE_VOLUME_DIR, exist_ok=True)
battle_model.save_model(f"{BATTLE_VOLUME_DIR}/m_overtake_classifier.json")

### Not included

- Track-limits / repeat-offender risk (needs more `f1_race_control_messages` backfill).
- Live wiring and DRS/telemetry features: FastF1 lap data has no DRS flag, so "within DRS range" is never claimed; the UI says "under 1 s at the end of the lap".